In [1]:
import os
import pandas as pd
import numpy as np
import pyodbc
from datetime import datetime, timedelta
import os
import json
from Functions import return_jalali_date, user_pyodbc_connection, pyodbc_connection
from Functions import Node1va2_Username, Node1va2_Password
from Functions import *


pd.set_option('display.max_rows', None)
pd.set_option('display.max_columns', None)
os.chdir(r'D:\Projects\Export Dashboard')


Read Data

In [2]:
NewExport = pd.read_excel(r'''\\172.31.50.113\Sale & Distribution\NewExport.xlsx''' , sheet_name=  'CurrencyParityRate')
nonsys = pd.read_csv(r'''\\172.31.50.113\Sale & Distribution\NonSysyemiForPowerBI.txt''', sep=',')
FactExcelAggr = pd.read_excel(r'\\172.31.50.113\Sale & Distribution\FactExcelAggr.xlsx') #FactExcelAggr + Khazane = nonsys

# Preprocess 

In [3]:
dict = {'دینار' : '107',
        'افغاني' : '103',
         'دلار امريكا' : '101',
          'ريال' : '108',
           'يورو' : '102',
            'روبل' : '106' }

dict1 = { '107': 'IraqDinar',
         '103':'Afghani',
         '101': 'USD',
         '108': 'Rial',
         '102':'Euro',
         '106': 'RUB'
}

dict2 = {'ريال' : '108',
        'Afghani' : '103',
        'IraqiDinar':'107',
         'USD' : '101',
          'Rial' : '108',
           'Euro' : '102',
            'RUB' : '106',
             'Dirham': '109',
              'TL': '104',
               'CAD':'105' }

In [4]:
nonsys = nonsys[nonsys['TableInfo'] == 'JameMali'] 

nonsys['NetAmount'] = nonsys['NetAmount']/1000000
FactExcelAggr['Amount_Net'] = FactExcelAggr['Amount_Net']/1000000

# MainDates

In [5]:
nonsys['MainDate'].astype('str')
nonsys['CurrencyCode'] = nonsys['CurrencyDesc'].map(dict2)
nonsys['MainDate'] = nonsys['MainDate'].astype('str')
nonsys['YearMonth'] = nonsys['MainDate'].str[:6]
nonsys['YearMonth']=nonsys['YearMonth'].astype('str')

NewExport['YearMonth'] = NewExport['YearMonth'].astype(str)
NewExport['YearMonth'] = NewExport['YearMonth'].astype("string")
NewExport['CurrencyID'] = NewExport['CurrencyID'].astype("string")


FactExcelAggr['CurrencyCode'] = FactExcelAggr['CurrencyDesc'].map(dict2)
FactExcelAggr['Main_Date'] = FactExcelAggr['Main_Date'].astype('str')
FactExcelAggr['YearMonth'] = FactExcelAggr['Main_Date'].str[:6]
FactExcelAggr = FactExcelAggr[FactExcelAggr['Flag_InOut'] == 2]


FactExcelAggr = FactExcelAggr[FactExcelAggr['YearMonth'] >= '140300']
nonsys = nonsys[nonsys['YearMonth'] >= '140300']


In [6]:
#Test 
print (Node1va2_Username)

ai


In [7]:
# DataQuality Connection
connection, cursor = user_pyodbc_connection("172.31.31.29", "DataQuality")

# Connection string to DataWarehouse DataBase / Node2
execute_connection, execute_cursor = user_pyodbc_connection(server="172.31.50.98",
                                                            database_name="SaleIntegratedModel",
                                                            username=Node1va2_Username,
                                                            password=Node1va2_Password)


today_date = datetime.now()
current_time = str(today_date.time())
persian_today_date = return_jalali_date(today_date)[0]
year = 1403

In [8]:
companies_App_dict = {3: {'CompanyName': 'گلرنگ سیستم', 'ApplicationCode': 1},
                      4: {'CompanyName': 'حمل و نقل گلرنگ ترابر', 'ApplicationCode': 1}, 
                      8: {'CompanyName': 'صنایع الکترونیک گلرنگ', 'ApplicationCode': 1}, 
                     13: {'CompanyName': 'آریان تجارت شرق', 'ApplicationCode': 1}, 
                     15: {'CompanyName': 'گل پخش اول', 'ApplicationCode': 1}, 
                     16: {'CompanyName': 'گروه صنعتی پاکشو', 'ApplicationCode': [1,4]}, 
                     17: {'CompanyName': 'صنایع سلولزی مارینا سان', 'ApplicationCode': 3}, 
                     18: {'CompanyName': 'پاکان پلاستکار', 'ApplicationCode': 1}, 
                     19: {'CompanyName': 'ماستر فوده', 'ApplicationCode': 1}, 
                     20: {'CompanyName': 'ایراندار', 'ApplicationCode': 1}, 
                     21: {'CompanyName': 'مارینا پخش هستی', 'ApplicationCode': 1}, 
                     24: {'CompanyName': 'افق توسعه معادن خاورمیانه', 'ApplicationCode': 1}, 
                     27: {'CompanyName': 'ویرا سیستم پویا', 'ApplicationCode': 1}, 
                     33: {'CompanyName': 'توسعه صنعتی سرافراز پارس (توسعه صنعتی سولفور کوب پارس (اسپیدکو))', 'ApplicationCode': 1}, 
                     34: {'CompanyName': 'توسعه فرآورده های نفتی افق خاورمیانه (موپیکو)', 'ApplicationCode': 1}, 
                     37: {'CompanyName': 'پاکان بسپار آرین', 'ApplicationCode': 3}, 
                     39: {'CompanyName': 'آریان کیمیا تک', 'ApplicationCode': [1,3]}, 
                     41: {'CompanyName': 'آزما نانو سیستم', 'ApplicationCode': 3}, 
                     42: {'CompanyName': 'فروشگاه های زنجیره ای افق کوروش', 'ApplicationCode': 5}, 
                     43: {'CompanyName': 'گلرنگ پخش', 'ApplicationCode': 1}, 
                     45: {'CompanyName': 'صنعت غذایی کورش', 'ApplicationCode': [5,1]}, 
                     47: {'CompanyName': 'فروشگاه های زنجیره ای فامیلی مدرن', 'ApplicationCode': [13,5]}, 
                     48: {'CompanyName': 'سامان پویش تامین (اسپات)', 'ApplicationCode': [1,11]}, 
                     51: {'CompanyName': 'کشت و صنعت برنج کوروش', 'ApplicationCode': 1}, 
                     54: {'CompanyName': 'صنعت خشکبار و حبوبات کوروش', 'ApplicationCode': 1}, 
                     55: {'CompanyName': 'هستی آرین تامین', 'ApplicationCode': [1,2]}, 
                     56: {'CompanyName': 'طلای ناب کوروش', 'ApplicationCode': 1}, 
                     59: {'CompanyName': 'پخش پدیده پایدار', 'ApplicationCode': 1}, 
                     60: {'CompanyName': 'سلامت پخش هستی', 'ApplicationCode': 9}, 
                     61: {'CompanyName': 'پدیده شیمی قرن', 'ApplicationCode': 1}, 
                     62: {'CompanyName': 'تولیدی فاران شیمی تویسرکان', 'ApplicationCode': 1}, 
                     63: {'CompanyName': 'پدیده شیمی غرب', 'ApplicationCode': [1,2,10]}, 
                     64: {'CompanyName': 'آرین سلامت سینا', 'ApplicationCode': [1,2,10]}, 
                     65: {'CompanyName': 'سپهر پلاستیک پدیده', 'ApplicationCode': 1}, 
                     66: {'CompanyName': 'پدیده شیمی جم', 'ApplicationCode': 1}, 
                     70: {'CompanyName': 'هستی آریا شیمی', 'ApplicationCode': 1}, 
                     71: {'CompanyName': 'آروین پلاست هکمتانه', 'ApplicationCode': 1}, 
                     73: {'CompanyName': 'مایا زیست فرآیند', 'ApplicationCode': 1}, 
                     74: {'CompanyName': 'ابیان دارو (هستی آرین دارو)', 'ApplicationCode': 1}, 
                     82: {'CompanyName': 'دنیا اینترنشنال گروپ', 'ApplicationCode': [1,4]}, 
                     83: {'CompanyName': 'تحقیقاتی و تولیدی واریان فارمد (واریان دارو پژوه)', 'ApplicationCode': 1}, 
                     88: {'CompanyName': 'خدمات تحقیقات آرین گستر', 'ApplicationCode': 1}, 
                     90: {'CompanyName': 'گلبرگ غذایی کوروش', 'ApplicationCode': 1}, 
                    107: {'CompanyName': 'ستاره طلایی سینا', 'ApplicationCode': [1,20]}, 
                    112: {'CompanyName': 'پدیده پایدار صنعت ساختمان (پدیده شیمی آرین)', 'ApplicationCode': 1}, 
                    113: {'CompanyName': 'فرآورده های پروتئینی کورش', 'ApplicationCode': 1}, 
                    116: {'CompanyName': 'سروش مانا فارمد', 'ApplicationCode': 1}, 
                    117: {'CompanyName': 'گروه صنایع غذایی پاکبان', 'ApplicationCode': 1}, 
                    119: {'CompanyName': 'دارانامهر', 'ApplicationCode': 1}, 
                    127: {'CompanyName': 'کالا رسان هستی (کوروش پخش)', 'ApplicationCode': 1}, 
                    135: {'CompanyName': 'پاکان به شو', 'ApplicationCode': 4}, 
                    149: {'CompanyName': 'آذر زیست دارو (فاران فارمد، شایان سلامت پارسه)', 'ApplicationCode': 1}, 
                    150: {'CompanyName': 'پدیده زیستی نانو', 'ApplicationCode': 1}, 
                    151: {'CompanyName': 'آوین شیمی پلاست', 'ApplicationCode': 1}, 
                    153: {'CompanyName': 'فرآورده های غذایی پروشات کوروش', 'ApplicationCode': 1}, 
                    156: {'CompanyName': 'گلرنگ ترابر قزوین', 'ApplicationCode': 1}, 
                    159: {'CompanyName': 'دنیا پخش افغانستان', 'ApplicationCode': [1,4]}, 
                    160: {'CompanyName': 'دام و طیور کوروش', 'ApplicationCode': 5}, 
                    167: {'CompanyName': 'افق الکوثر عراق', 'ApplicationCode': 5}, 
                    190: {'CompanyName': 'صنایع غذایی کیلوس', 'ApplicationCode': [1,10]}, 
                    191: {'CompanyName': 'هستی الکتریک تارا', 'ApplicationCode': 1}, 
                    194: {'CompanyName': 'صنایع بهداشتی پایدار', 'ApplicationCode': 1}, 
                    196: {'CompanyName': 'آرین تجارت رایحه آفرین', 'ApplicationCode': 1}, 
                    197: {'CompanyName': 'گندم طلایی کوروش', 'ApplicationCode': 5}, 
                    200: {'CompanyName': 'نوین زعفران', 'ApplicationCode': [1,11]}, 
                    201: {'CompanyName': 'آمیزه های پلیمری سپهر پاک', 'ApplicationCode': [1,10]}, 
                    202: {'CompanyName': 'فارمد سلامت سینا', 'ApplicationCode': 1}, 
                    204: {'CompanyName': 'پژوهش گستران تغذیه آسان', 'ApplicationCode': 1}, 
                    206: {'CompanyName': 'گروه صنعتی سورنا سلولز', 'ApplicationCode': 3}, 
                    212: {'CompanyName': 'شیرینی و شکلات کوروش', 'ApplicationCode': 1}, 
                    274: {'CompanyName': 'پارس فیلم نت', 'ApplicationCode': 3}, 
                    277: {'CompanyName': 'آریو سلولز خزر', 'ApplicationCode': 3}, 
                    357: {'CompanyName': 'سپید ماکیان', 'ApplicationCode': 11}, 
                    406: {'CompanyName': 'ابیان فارمد', 'ApplicationCode': 1}, 
                    413: {'CompanyName': 'سوئیس رز عراق', 'ApplicationCode': 22}, 
                    414: {'CompanyName': 'جویا بهنود', 'ApplicationCode': 1}
                    }


companies_JameMali_dict = {2: {'CompanyName': 'مجتمع تجاری فرهنگی کوروش', 'ApplicationCode': -1, 'Moein': [611133,611144,641171]},
                           6: {'CompanyName': 'سرزمین بازی کوروش', 'ApplicationCode': -1, 'Moein': [611131]},
                           7: {'CompanyName': 'پردیس سینمایی کوروش', 'ApplicationCode': -1, 'Moein': [611131,611132]},
                          20: {'CompanyName': 'ایراندار', 'ApplicationCode': -1, 'Moein': [611141]},
                          22: {'CompanyName': 'آرین سلولز صنعت', 'ApplicationCode': -1, 'Moein': [611133,611131]},
                          30: {'CompanyName': 'مهندسین مشاور افق معادن خاورمیانه', 'ApplicationCode': -1, 'Moein': [611131,641181]},
                          53: {'CompanyName': 'واسپاری ارزش آفرین گلرنگ (لیزینگ)', 'ApplicationCode': -1, 'Moein': [611162,611163,641153,641154]},
                          62: {'CompanyName': 'فاران شیمی', 'ApplicationCode': -1, 'Moein': [611141]},
                          65: {'CompanyName': 'سپهر پلاستیک پدیده', 'ApplicationCode': -1, 'Moein': [611141]},
                         118: {'CompanyName': 'تارا', 'ApplicationCode': -1, 'Moein': [641131,611131,641158,611111,631111]},
                         128: {'CompanyName': 'بیمه سامان', 'ApplicationCode': -1, 'Moein': [611131]},
                         136: {'CompanyName': 'چیکن فامیلی', 'ApplicationCode': -1, 'Moein': [611111,631111]},
                         145: {'CompanyName': 'سبد گردان کوروش', 'ApplicationCode': -1, 'Moein': [611163]},
                         162: {'CompanyName': 'کارکیا سورنا', 'ApplicationCode': -1, 'Moein': [611131]},
                         176: {'CompanyName': 'مریدین روسیه', 'ApplicationCode': -1, 'Moein': [611111,621111,631111]},
                         186: {'CompanyName': 'خدمات خودرویی سامیا', 'ApplicationCode': -1, 'Moein': [611111,611131,611133,631111,621111,641131]},
                         187: {'CompanyName': 'مینیمم مارکت', 'ApplicationCode': -1, 'Moein': [611111,621111,631111]},
                         192: {'CompanyName': 'امین پدیدار', 'ApplicationCode': -1, 'Moein': [611111,611131,621111,631111]},
                         211: {'CompanyName': 'غذای سالم هستی', 'ApplicationCode': -1, 'Moein': [611111,611131,631111]},
                         212: {'CompanyName': 'شیرینی و شکلات کوروش', 'ApplicationCode': -1, 'Moein': [611141,621111]},
                         259: {'CompanyName': 'فاوا فناوری افق', 'ApplicationCode': -1, 'Moein': [611111,611131]},
                         265: {'CompanyName': 'فروتل', 'ApplicationCode': -1, 'Moein': [611131,621131,631111]},
                         266: {'CompanyName': 'خوان گستر هستی', 'ApplicationCode': -1, 'Moein': [611111,621111]},
                         282: {'CompanyName': 'فرا تجارت زردکوه', 'ApplicationCode': -1, 'Moein': [611111]},
                         288: {'CompanyName': 'آرین تامین آفرین', 'ApplicationCode': -1, 'Moein': [611163,611131]},
                         336: {'CompanyName': 'نوین پوش هستی', 'ApplicationCode': -1, 'Moein': [611111]},
                         340: {'CompanyName': 'آی پوش', 'ApplicationCode': -1, 'Moein': [611111,621111,631111]},
                         353: {'CompanyName': 'غذای ناب کوروش', 'ApplicationCode': -1, 'Moein': [611111]},
                         }

### Gathering Data from SaleIntegratedModel..factsalealk

In [9]:
ALK_rows = []

Alk_query = """
select CompanyCode ,  round((SUM(case when InvoceType = 2 then NetAmount else 0 end) - SUM(case when InvoceType = 3 then NetAmount else 0 end))/ 1000000.0, 0) as SumQatE
from SaleIntegratedModel..factsalealk
where SaleType = 2 and left(maindate , 6) >= 140300
group by CompanyCode"""



try:
    execute_cursor.execute(Alk_query)
    ALK_total_rows = execute_cursor.fetchall()
    
    for row in ALK_total_rows:
        if not row[0]:
            row[0] = "Null"
        if not row[1]:
            row[1] = 0
            
        CompanyCode, ALK_total = row
        ALK_rows.append(
            {
                "CompanyCode": CompanyCode,
                "ALK_total": ALK_total
            }
        )
except Exception as e:
     print("Executing iframe_sql query failed", e)

ALK_df = pd.DataFrame(ALK_rows)

ALK_df = ALK_df[[ "CompanyCode", 'ALK_total']]

In [10]:
ALK_df

,CompanyCode,ALK_total
0,167,21189605.000000


### Gathering Data from SaleIntegratedModel..FactSaleNonDistribut

In [11]:
Nondistribute_rows = []

NondistributeCompanies_query = """
select CompanyCode ,  round((SUM(case when InvoceType = 2 then NetAmount else 0 end) - SUM(case when InvoceType = 3 then NetAmount else 0 end))/ 1000000.0, 0) as SumQatE
from SaleIntegratedModel..FactSaleNonDistribut  where SaleType = 2 and left(maindate , 6) >= 140300
group by CompanyCode """



try:
    execute_cursor.execute(NondistributeCompanies_query)
    Nondistribute_total_rows = execute_cursor.fetchall()
    
    for row in Nondistribute_total_rows:
        if not row[0]:
            row[0] = "Null"
        if not row[1]:
            row[1] = 0
            
        CompanyCode, Nondistribute_Total = row
        Nondistribute_rows.append(
            {
                "CompanyCode": CompanyCode,
                "Nondistribute_Total": Nondistribute_Total
            }
        )
except Exception as e:
     print("Executing iframe_sql query failed", e)

Nondistribute_df = pd.DataFrame(Nondistribute_rows)

Nondistribute_df = Nondistribute_df[[ "CompanyCode", 'Nondistribute_Total']]

# Merge All sources Data

In [12]:
#preparing nonsys data
 
TotalSale_Mali_and_excel_df = nonsys[['CompanyCode','NetAmount']].groupby(by = 'CompanyCode').sum().reset_index()
FactExcelAggr_df = FactExcelAggr[['CompanyCode' , 'Amount_Net']].groupby(by = 'CompanyCode').sum().reset_index()

unifying Column names

In [13]:
Nondistribute_df.columns = ['CompanyCode' , 'N_NetAmount']
TotalSale_Mali_and_excel_df.columns = ['CompanyCode' , 'O_NetAmount']
FactExcelAggr_df.columns = ['CompanyCode' , 'X_NetAmount']
ALK_df.columns = ['CompanyCode' , 'ALK_NetAmount']

In [14]:
#Merge Nondistribute / Excel and khazane

Total_rows = pd.concat([Nondistribute_df , TotalSale_Mali_and_excel_df , FactExcelAggr_df] , axis = 0) 
Total_rows.TotalSale = Total_rows.N_NetAmount.astype(float)
Total_rows = Total_rows.groupby(by = 'CompanyCode').sum().reset_index()

print(f'''the total_rows is {len(Total_rows)} and the number of Companies is {Total_rows.CompanyCode.nunique()}''')

the total_rows is 30 and the number of Companies is 30


C:\Users\Soleimani.Yeganeh\AppData\Local\Temp\ipykernel_23576\2298988150.py:4: UserWarning: Pandas doesn't allow columns to be created via a new attribute name - see https://pandas.pydata.org/pandas-docs/stable/indexing.html#attribute-access
  Total_rows.TotalSale = Total_rows.N_NetAmount.astype(float)


In [15]:
nonsys_df_Net = TotalSale_Mali_and_excel_df[['CompanyCode' , 'O_NetAmount']].groupby(by = ['CompanyCode']).sum().reset_index()

In [16]:
nonsys_df_Net

,CompanyCode,O_NetAmount
0,20,6.379968e+05
1,62,1.550180e+06
2,65,1.068066e+05
3,176,5.357842e+06
4,187,1.856030e+06
5,190,9.924568e+03
6,212,2.205036e+05


In [17]:
Total_rows_Net_tot = pd.merge(Nondistribute_df , nonsys_df_Net ,how = 'outer' , right_on= ['CompanyCode'] , left_on =  ['CompanyCode']) #merge nonsys and nondistribute

Total_rows_Net_tot_final = pd.merge(Total_rows_Net_tot , FactExcelAggr_df , how = 'outer' ,right_on= ['CompanyCode'] , left_on =  ['CompanyCode'])

In [18]:
Total_rows_Net_tot_final = Total_rows_Net_tot_final[['N_NetAmount', 'O_NetAmount', 'X_NetAmount','CompanyCode']].groupby(by = 'CompanyCode').sum().reset_index()

FactIfrmae

In [19]:
# -------------------------------------------------- Iframe Amount per Companay

iframe_rows = []

iframe_sql = """
SELECT CompanyCode, dim.Company_PName,
            ROUND((SUM(CASE WHEN Sale_State = 2 THEN ROUND(Amount_Net, 0) ELSE 0 END) -
                   SUM(CASE WHEN Sale_State = 3 THEN ROUND(Amount_Net, 0) ELSE 0 END)) / 1000000.0, 0) AS Iframe_Total
FROM Qlikview.SSAS_Aggr.FactSaleIFrame as fact WITH (NOLOCK)
LEFT JOIN QlikView.SSAS_Aggr.Company as dim WITH (NOLOCK) ON fact.CompanyCode = dim.[Master Code]
WHERE Flag_InOut = 2 and left(Main_Date , 6) >=140300
GROUP BY CompanyCode, dim.Company_PName;
"""

try:
    execute_cursor.execute(iframe_sql)
    iframe_total_rows = execute_cursor.fetchall()
    
    for row in iframe_total_rows:
        if not row[1]:
            row[1] = "Null"
        if not row[0]:
            row[0] = "Null"
        if not row[2]:
            row[2] = 0
            
        CompanyCode, CompanyName, Iframe_Total = row
        iframe_rows.append(
            {
                "CompanyName": CompanyName,
                "CompanyCode": CompanyCode,
                #"Main_Date" : Main_Date,
                "Iframe_Total": Iframe_Total

            }
        )
except Exception as e:
     print("Executing iframe_sql query failed", e)

iframe_df = pd.DataFrame(iframe_rows)



iframe_df = iframe_df[["CompanyName", "CompanyCode",  "Iframe_Total"]]

In [20]:
iframe_df_Net = iframe_df[['CompanyName' , 'CompanyCode' , 'Iframe_Total']]
iframe_df_Net = iframe_df_Net.groupby(by = ['CompanyName'	,'CompanyCode']).sum().reset_index()

In [21]:
whole_final = pd.merge(iframe_df_Net , Total_rows_Net_tot_final ,how = 'outer' , right_on= ['CompanyCode'] , left_on =  ['CompanyCode'])
whole_final = pd.merge(whole_final , ALK_df ,how = 'outer' , right_on= ['CompanyCode'] , left_on =  ['CompanyCode'])


whole_final.N_NetAmount = whole_final.N_NetAmount.fillna(0)
whole_final.O_NetAmount = whole_final.O_NetAmount.fillna(0)
whole_final.X_NetAmount = whole_final.X_NetAmount.fillna(0)
whole_final.ALK_NetAmount = whole_final.ALK_NetAmount.fillna(0)



whole_final['Iframe_Total'] = whole_final['Iframe_Total'].astype('int')
whole_final['N_NetAmount'] = whole_final['N_NetAmount'].astype('int')
whole_final['X_NetAmount'] = whole_final['X_NetAmount'].astype('int')
whole_final['O_NetAmount'] = whole_final['O_NetAmount'].astype('int')
whole_final['ALK_NetAmount'] = whole_final['ALK_NetAmount'].astype('int')





whole_final['TotalSale_Whole'] = whole_final['N_NetAmount'] + whole_final['O_NetAmount'] + whole_final['X_NetAmount'] + whole_final['ALK_NetAmount']
whole_final.drop(['N_NetAmount' , 'O_NetAmount' ,'X_NetAmount' ] , axis = 1 , inplace = True)



whole_final['Differenece'] = whole_final['Iframe_Total'] - whole_final['TotalSale_Whole']
whole_final

,CompanyName,CompanyCode,Iframe_Total,ALK_NetAmount,TotalSale_Whole,Differenece
0,گروه صنعتی پاکشو,16,21908937,0,21908937,0
1,صنایع سلولزی مارینا سان,17,77283,0,77283,0
2,پاکان پلاستکار,18,731254,0,731254,0
3,صنایع غذایی ماستر فوده,19,1069107,0,1069107,0
4,ایراندار,20,637997,0,637996,1
5,آریان کیمیا تک,39,2529473,0,2529473,0
6,سامان پویش تامین (اسپات),48,4084775,0,4084775,0
7,صنعت خشکبار و حبوبات کوروش,54,1624627,0,1624627,0
8,هستی آرین تامین,55,158680,0,158680,0
9,پدیده شیمی قرن,61,7846098,0,7846098,0


In [22]:
df_anomalies = whole_final[whole_final['Differenece'] > 10]
df_anomalies.drop(['TotalSale_Whole' , 'ALK_NetAmount' , 'Iframe_Total'] ,axis = 1 ,  inplace = True)
df_anomalies.columns = ['نام شرکت' , 'کد مستر شرکت' , 'اختلاف']

C:\Users\Soleimani.Yeganeh\AppData\Local\Temp\ipykernel_23576\1384566625.py:2: SettingWithCopyWarning: 
A value is trying to be set on a copy of a slice from a DataFrame

See the caveats in the documentation: https://pandas.pydata.org/pandas-docs/stable/user_guide/indexing.html#returning-a-view-versus-a-copy
  df_anomalies.drop(['TotalSale_Whole' , 'ALK_NetAmount' , 'Iframe_Total'] ,axis = 1 ,  inplace = True)


In [23]:
import requests
import json

if not df_anomalies.empty:
    payload = json.dumps({

        "token": "P880P2HLA6MO71PWTXTWR",
        "providerId": 6,
        "sendType": 0,
        "email": 'soleimani.yeganeh@golrang.com',
        "ccRecipients": ['Aghdasifam.Masoud@Golrang.com'],
        "subject": "Alert: KPI Values Exceeded!",
        "body": f"موارد زیر مغایرت دارند: \n\n {df_anomalies.to_html(index=False, border=1, justify='center')}",
        "fileAttachmentAddress": ""
    })
    headers = {'Content-Type': 'application/json'}
    response = requests.post("https://Esp-api.gig.services/email/saveEmail", headers=headers, data=payload)
    print(response.status_code, response.text)
else:
    payload = json.dumps({
        "token": "P880P2HLA6MO71PWTXTWR",
        "providerId": 6,
        "sendType": 0,
        "email":'soleimani.yeganeh@golrang.com',
        "ccRecipients": ['Aghdasifam.Masoud@Golrang.com'], #It should be in []
        "subject": "Confirmation of Descending program execution!",
        "body": "انجام شد",
        "fileAttachmentAddress": ""
    })
    headers = {'Content-Type': 'application/json'}
    response = requests.post("https://Esp-api.gig.services/email/saveEmail", headers=headers, data=payload)
    print(response.status_code, response.text)



200 {"success":true,"message":null,"data":"7763cc84-a07a-f011-87d4-00505690dbb7"}
